In [54]:
from pathlib import Path
from datetime import datetime, timedelta
import time, json
import requests
import pandas as pd

RAW_DIR = Path(r"C:\Users\whoeltermann\OneDrive\Sample Works\Python\FRED_environmental_projects\data\raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

COOPS_DATA_API = "https://api.tidesandcurrents.noaa.gov/api/prod/datagetter"
COOPS_STATIONS_FS_QUERY = "https://mapservices.weather.noaa.gov/static/rest/services/NOS_Observations/CO_OPS_Stations/FeatureServer/1/query"

In [55]:
# Check stations in south florida

def list_florida(layer):
    url = f"{BASE}/{layer}/query"
    params = {
        "where": "state='FL'",
        "outFields": "id,name,latitude,longitude",
        "returnGeometry": "false",
        "f": "json",
        "resultRecordCount": 2000
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    feats = r.json().get("features", [])
    return feats

fl_layer0 = list_florida(0)
len(fl_layer0)

import pandas as pd

df_fl = pd.DataFrame([f["attributes"] for f in fl_layer0])
df_fl.sort_values("id").head(25)

,id,name,latitude,longitude
0,8720030,Fernandina Beach,30.671356,-81.465840
1,8720218,Mayport (Bar Pilots Dock),30.398167,-81.427890
2,8720219,Dames Point,30.386700,-81.558300
3,8720226,"Southbank Riverwalk, St Johns River",30.320000,-81.658300
4,8720357,I-295 Buckman Bridge,30.192417,-81.688890
5,8721604,"Trident Pier, Port Canaveral",28.415800,-80.593100
6,8722670,"Lake Worth Pier, Atlantic Ocean",26.612778,-80.034164
7,8722956,South Port Everglades,26.081667,-80.116670
8,8723214,Virginia Key,25.731400,-80.161800
9,8723970,"Vaca Key, Florida Bay",24.711000,-81.106500


In [56]:
df_fl = pd.DataFrame([f["attributes"] for f in fl_layer0])

# Keep only needed columns
df_fl = df_fl[["id", "name", "latitude", "longitude"]]

df_fl.head()
xmin = df_fl["longitude"].min()
xmax = df_fl["longitude"].max()
ymin = df_fl["latitude"].min()
ymax = df_fl["latitude"].max()

# South Florida bbox (lon/lat): xmin, ymin, xmax, ymax
BBOX = (xmin, ymin, xmax, ymax)

BBOX

(np.float64(-87.2112),
 np.float64(24.5557),
 np.float64(-80.034164),
 np.float64(30.671356))

In [57]:
# review bbox with visual inspection
lon_min, lat_min, lon_max, lat_max = BBOX
inside_bbox = df_fl[
    (df_fl["longitude"] >= lon_min) &
    (df_fl["longitude"] <= lon_max) &
    (df_fl["latitude"]  >= lat_min) &
    (df_fl["latitude"]  <= lat_max)
].copy()
plot_df = inside_bbox.rename(columns={
    "id": "station_id",
    "name": "station_name",
    "latitude": "lat",
    "longitude": "lon"
})[["station_id","station_name","lat","lon"]]

# mapping bbox
map_bbox_and_stations(BBOX, plot_df, tiles="CartoDB positron")

In [58]:
# Raw pull settings
PRODUCT = "water_level"
DATUM = "MSL"
UNITS = "metric"
TIME_ZONE = "gmt"
INTERVAL = "6"      # 6-minute
CHUNK_DAYS = 31     # safe for 6-min water_level

BEGIN = "20100101"  # YYYYMMDD
END   = "20101231"

In [59]:
print("stations_df:", len(stations_df))
print("inside_bbox:", len(inside_bbox))

stations_df: 411
inside_bbox: 23


In [60]:
def discover_active_water_level_stations(bbox):
    params = {
        "where": "1=1",
        "geometry": f"{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}",
        "geometryType": "esriGeometryEnvelope",
        "inSR": 4326,
        "spatialRel": "esriSpatialRelIntersects",
        "outFields": "*",
        "returnGeometry": "false",
        "f": "json",
        "resultRecordCount": 2000,
    }
    r = requests.get(COOPS_STATIONS_FS_QUERY, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()
    feats = data.get("features", [])
    rows = [f.get("attributes", {}) for f in feats]
    df = pd.DataFrame(rows)

    # normalize column names (service fields vary slightly)
    def pick(*cands):
        for c in cands:
            if c in df.columns: return c
        return None

    c_id  = pick("id", "stationid", "station_id", "nos_id")
    c_nm  = pick("name", "stationname", "station_name")
    c_lat = pick("latitude", "lat")
    c_lon = pick("longitude", "lon", "long")
    c_st  = pick("state")

    if not all([c_id, c_nm, c_lat, c_lon]):
        raise RuntimeError(f"Unexpected station fields. Got columns: {list(df.columns)}")

    out = df[[c_id, c_nm, c_lat, c_lon] + ([c_st] if c_st else [])].copy()
    out.columns = ["station_id", "station_name", "lat", "lon"] + (["state"] if c_st else [])
    if "state" in out.columns:
        out = out[out["state"].fillna("").str.upper().eq("FL")]

    out["station_id"] = out["station_id"].astype(str)
    out = out.drop_duplicates(subset=["station_id"]).sort_values("station_name").reset_index(drop=True)
    return out

stations_df = discover_active_water_level_stations(BBOX)
len(stations_df), stations_df.head(10)

(650,
   station_id                  station_name      lat      lon state
 0    8723453                     ADAMS KEY  25.3967 -80.2333    FL
 1    8724992                   ADDISON BAY  25.9633 -81.6650    FL
 2    8726614            ALAFIA RIVER NORTH  27.8700 -82.3283    FL
 3    8729015            ALLANTON, EAST BAY  30.0300 -85.4650    FL
 4    8729364                ALLAQUAY BAYOU  30.4883 -86.2050    FL
 5    8721374     ALLENHURST,HAULOVER CANAL  28.7333 -80.7567    FL
 6    8729152  ALLIGATOR BAYOU, PANAMA CITY  30.1700 -85.7550    FL
 7    8728255              ALLIGATOR HARBOR  29.9117 -84.3650    FL
 8    8728288               ALLIGATOR POINT  29.9033 -84.4133    FL
 9    8728261      ALLIGATOR POINT, SW CAPE  29.8950 -84.3883    FL)

In [61]:
def discover_active_water_level_stations(bbox):
    params = {
        "where": "1=1",
        "geometry": f"{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}",
        "geometryType": "esriGeometryEnvelope",
        "inSR": 4326,
        "spatialRel": "esriSpatialRelIntersects",
        "outFields": "*",
        "returnGeometry": "false",
        "f": "json",
        "resultRecordCount": 2000,
    }
    r = requests.get(COOPS_STATIONS_FS_QUERY, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()
    feats = data.get("features", [])
    rows = [f.get("attributes", {}) for f in feats]
    df = pd.DataFrame(rows)

    # normalize column names
    def pick(*cands):
        for c in cands:
            if c in df.columns: return c
        return None

    c_id  = pick("id", "stationid", "station_id", "nos_id")
    c_nm  = pick("name", "stationname", "station_name")
    c_lat = pick("latitude", "lat")
    c_lon = pick("longitude", "lon", "long")
    c_st  = pick("state")

    if not all([c_id, c_nm, c_lat, c_lon]):
        raise RuntimeError(f"Unexpected station fields. Got columns: {list(df.columns)}")

    out = df[[c_id, c_nm, c_lat, c_lon] + ([c_st] if c_st else [])].copy()
    out.columns = ["station_id", "station_name", "lat", "lon"] + (["state"] if c_st else [])
    if "state" in out.columns:
        out = out[out["state"].fillna("").str.upper().eq("FL")]

    out["station_id"] = out["station_id"].astype(str)
    out = out.drop_duplicates(subset=["station_id"]).sort_values("station_name").reset_index(drop=True)
    return out

stations_df = discover_active_water_level_stations(BBOX)
len(stations_df), stations_df.head(10)

(650,
   station_id                  station_name      lat      lon state
 0    8723453                     ADAMS KEY  25.3967 -80.2333    FL
 1    8724992                   ADDISON BAY  25.9633 -81.6650    FL
 2    8726614            ALAFIA RIVER NORTH  27.8700 -82.3283    FL
 3    8729015            ALLANTON, EAST BAY  30.0300 -85.4650    FL
 4    8729364                ALLAQUAY BAYOU  30.4883 -86.2050    FL
 5    8721374     ALLENHURST,HAULOVER CANAL  28.7333 -80.7567    FL
 6    8729152  ALLIGATOR BAYOU, PANAMA CITY  30.1700 -85.7550    FL
 7    8728255              ALLIGATOR HARBOR  29.9117 -84.3650    FL
 8    8728288               ALLIGATOR POINT  29.9033 -84.4133    FL
 9    8728261      ALLIGATOR POINT, SW CAPE  29.8950 -84.3883    FL)

In [62]:
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
stations_csv  = RAW_DIR / f"noaa_coops_southfl_active_stations_{stamp}.csv"
stations_json = RAW_DIR / f"noaa_coops_southfl_active_stations_{stamp}.json"

stations_df.to_csv(stations_csv, index=False)
stations_json.write_text(stations_df.to_json(orient="records", indent=2), encoding="utf-8")

stations_csv

WindowsPath('C:/Users/whoeltermann/OneDrive/Sample Works/Python/FRED_environmental_projects/data/raw/noaa_coops_southfl_active_stations_20260223_130133.csv')

In [63]:
def parse_yyyymmdd(s): return datetime.strptime(s, "%Y%m%d")
def fmt_yyyymmdd(dt):  return dt.strftime("%Y%m%d")

def chunk_dates(begin_yyyymmdd, end_yyyymmdd, max_days):
    begin = parse_yyyymmdd(begin_yyyymmdd)
    end   = parse_yyyymmdd(end_yyyymmdd)
    cur = begin
    while cur <= end:
        chunk_end = min(end, cur + timedelta(days=max_days - 1))
        yield fmt_yyyymmdd(cur), fmt_yyyymmdd(chunk_end)
        cur = chunk_end + timedelta(days=1)

def coops_get_json(params, retries=4, timeout=60, backoff=1.0):
    last = None
    for i in range(retries):
        try:
            r = requests.get(COOPS_DATA_API, params=params, timeout=timeout)
            r.raise_for_status()
            data = r.json()
            if isinstance(data, dict) and "error" in data:
                raise RuntimeError(data["error"])
            return data
        except Exception as e:
            last = e
            time.sleep(backoff * (2 ** i))
    raise RuntimeError(f"NOAA request failed after {retries} tries: {params}") from last

In [64]:
def safe_name(s):
    return "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in s.strip())

def pull_station(station_id, station_name):
    out_dir = RAW_DIR / "noaa_coops" / f"{station_id}_{safe_name(station_name)}" / f"product={PRODUCT}_datum={DATUM}_tz={TIME_ZONE}"
    out_dir.mkdir(parents=True, exist_ok=True)

    all_rows = []
    for b, e in chunk_dates(BEGIN, END, CHUNK_DAYS):
        params = {
            "station": station_id,
            "product": PRODUCT,
            "begin_date": b,
            "end_date": e,
            "datum": DATUM,
            "units": UNITS,
            "time_zone": TIME_ZONE,
            "format": "json",
            "application": "FRED_environmental_projects",
        }
        if INTERVAL:
            params["interval"] = INTERVAL

        payload = coops_get_json(params)

        # raw JSON chunk
        chunk_path = out_dir / f"{station_id}_{b}_{e}.json"
        chunk_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

        # stack rows
        for obs in payload.get("data", []):
            all_rows.append({"station_id": station_id, "station_name": station_name, **obs})

    df = pd.DataFrame(all_rows)
    if not df.empty and "t" in df.columns:
        df["t"] = pd.to_datetime(df["t"], errors="coerce", utc=True)
        df = df.dropna(subset=["t"]).sort_values("t")

    combined_csv = out_dir / f"{station_id}_{safe_name(station_name)}_{PRODUCT}_{BEGIN}_{END}_combined.csv"
    df.to_csv(combined_csv, index=False)

    return str(out_dir), str(combined_csv), len(df)

results = []
for _, r in stations_df.iterrows():
    sid = str(r["station_id"])
    name = r["station_name"]
    try:
        out_dir, csv_path, n = pull_station(sid, name)
        print(f"[OK] {sid} {name}: {n} rows")
        results.append((sid, name, n, out_dir, csv_path))
    except Exception as e:
        print(f"[FAIL] {sid} {name}: {e}")
        results.append((sid, name, None, None, None))

results_df = pd.DataFrame(results, columns=["station_id","station_name","n_rows","out_dir","combined_csv"])
results_df

[FAIL] 8723453 ADAMS KEY: NOAA request failed after 4 tries: {'station': '8723453', 'product': 'water_level', 'begin_date': '20100101', 'end_date': '20100131', 'datum': 'MSL', 'units': 'metric', 'time_zone': 'gmt', 'format': 'json', 'application': 'FRED_environmental_projects', 'interval': '6'}
[FAIL] 8724992 ADDISON BAY: NOAA request failed after 4 tries: {'station': '8724992', 'product': 'water_level', 'begin_date': '20100101', 'end_date': '20100131', 'datum': 'MSL', 'units': 'metric', 'time_zone': 'gmt', 'format': 'json', 'application': 'FRED_environmental_projects', 'interval': '6'}
[FAIL] 8726614 ALAFIA RIVER NORTH: NOAA request failed after 4 tries: {'station': '8726614', 'product': 'water_level', 'begin_date': '20100101', 'end_date': '20100131', 'datum': 'MSL', 'units': 'metric', 'time_zone': 'gmt', 'format': 'json', 'application': 'FRED_environmental_projects', 'interval': '6'}
[FAIL] 8729015 ALLANTON, EAST BAY: NOAA request failed after 4 tries: {'station': '8729015', 'product

KeyboardInterrupt: 

In [ ]:
len(stations_df)

In [65]:
# Check 

def list_florida(layer):
    url = f"{BASE}/{layer}/query"
    params = {
        "where": "state='FL'",
        "outFields": "id,name,latitude,longitude",
        "returnGeometry": "false",
        "f": "json",
        "resultRecordCount": 2000
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    feats = r.json().get("features", [])
    return feats

fl_layer0 = list_florida(0)
len(fl_layer0)

import pandas as pd

df_fl = pd.DataFrame([f["attributes"] for f in fl_layer0])
df_fl.sort_values("latitude").head(20)

,id,name,latitude,longitude
10,8724580,Key West,24.555700,-81.807900
9,8723970,"Vaca Key, Florida Bay",24.711000,-81.106500
8,8723214,Virginia Key,25.731400,-80.161800
7,8722956,South Port Everglades,26.081667,-80.116670
11,8725114,"NAPLES BAY, NORTH",26.136700,-81.788300
6,8722670,"Lake Worth Pier, Atlantic Ocean",26.612778,-80.034164
12,8725520,Fort Myers,26.647778,-81.871110
13,8726384,Port Manatee,27.638700,-82.562100
14,8726520,St. Petersburg,27.760600,-82.626900
15,8726607,Old Port Tampa,27.857800,-82.552800


In [69]:
station_ids = list(df_fl["id"].astype(str).unique())

len(station_ids), station_ids[:23]

(23,
 ['8720030',
  '8720218',
  '8720219',
  '8720226',
  '8720357',
  '8721604',
  '8722670',
  '8722956',
  '8723214',
  '8723970',
  '8724580',
  '8725114',
  '8725520',
  '8726384',
  '8726520',
  '8726607',
  '8726674',
  '8726724',
  '8727520',
  '8728690',
  '8729108',
  '8729210',
  '8729840'])

In [73]:
from pathlib import Path

SF_ROOT = RAW_DIR / "noaa_coops_south_florida_active23"
SF_ROOT.mkdir(parents=True, exist_ok=True)

SF_ROOT

def pull_station_sf(station_id, station_name):
    out_dir = SF_ROOT / f"{station_id}_{station_name.replace(' ','_')}"
    out_dir.mkdir(parents=True, exist_ok=True)

    all_rows = []

    for b, e in chunk_dates(BEGIN, END, CHUNK_DAYS):
        params = {
            "station": station_id,
            "product": PRODUCT,
            "begin_date": b,
            "end_date": e,
            "datum": DATUM,
            "units": UNITS,
            "time_zone": TIME_ZONE,
            "format": "json",
            "application": "FRED_environmental_projects",
        }
        payload = coops_get_json(params)

        # save raw chunk
        (out_dir / f"{station_id}_{b}_{e}.json").write_text(
            json.dumps(payload, indent=2), encoding="utf-8"
        )

        for obs in payload.get("data", []):
            all_rows.append({
                "station_id": station_id,
                "station_name": station_name,
                **obs
            })

    df = pd.DataFrame(all_rows)

    combined_csv = out_dir / f"{station_id}_{BEGIN}_{END}_combined.csv"
    df.to_csv(combined_csv, index=False)

    return len(df)

In [74]:
results = []

for sid in station_ids:
    name = df_fl.loc[df_fl["id"].astype(str)==sid, "name"].values[0]
    try:
        n = pull_station_sf(sid, name)
        print(f"[OK] {sid} {name}: {n} rows")
        results.append((sid, name, n))
    except Exception as e:
        print(f"[FAIL] {sid}: {e}")
        results.append((sid, name, None))

results_df = pd.DataFrame(results, columns=["station_id","station_name","n_rows"])
results_df

[OK] 8720030 Fernandina Beach: 87600 rows
[OK] 8720218 Mayport (Bar Pilots Dock): 87600 rows
[FAIL] 8720219: NOAA request failed after 4 tries: {'station': '8720219', 'product': 'water_level', 'begin_date': '20100101', 'end_date': '20100131', 'datum': 'MSL', 'units': 'metric', 'time_zone': 'gmt', 'format': 'json', 'application': 'FRED_environmental_projects'}
[FAIL] 8720226: NOAA request failed after 4 tries: {'station': '8720226', 'product': 'water_level', 'begin_date': '20100101', 'end_date': '20100131', 'datum': 'MSL', 'units': 'metric', 'time_zone': 'gmt', 'format': 'json', 'application': 'FRED_environmental_projects'}
[OK] 8720357 I-295 Buckman Bridge: 87600 rows
[OK] 8721604 Trident Pier, Port Canaveral: 87600 rows
[FAIL] 8722670: NOAA request failed after 4 tries: {'station': '8722670', 'product': 'water_level', 'begin_date': '20100101', 'end_date': '20100131', 'datum': 'MSL', 'units': 'metric', 'time_zone': 'gmt', 'format': 'json', 'application': 'FRED_environmental_projects'}


,station_id,station_name,n_rows
0,8720030,Fernandina Beach,87600.0
1,8720218,Mayport (Bar Pilots Dock),87600.0
2,8720219,Dames Point,NaN
3,8720226,"Southbank Riverwalk, St Johns River",NaN
4,8720357,I-295 Buckman Bridge,87600.0
5,8721604,"Trident Pier, Port Canaveral",87600.0
6,8722670,"Lake Worth Pier, Atlantic Ocean",NaN
7,8722956,South Port Everglades,NaN
8,8723214,Virginia Key,87600.0
9,8723970,"Vaca Key, Florida Bay",85960.0


In [75]:
len(station_ids), station_ids[:23]

(23,
 ['8720030',
  '8720218',
  '8720219',
  '8720226',
  '8720357',
  '8721604',
  '8722670',
  '8722956',
  '8723214',
  '8723970',
  '8724580',
  '8725114',
  '8725520',
  '8726384',
  '8726520',
  '8726607',
  '8726674',
  '8726724',
  '8727520',
  '8728690',
  '8729108',
  '8729210',
  '8729840'])